# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 — Data Loading, Understanding & Exploration

**Student:** [Your Name] | **Reg No:** [Your Reg No]  
**Course:** DSA 8301 — Statistical Inference for Big Data  
**Lecturer:** Prof. Jacob Ong'ala  
**Institution:** Strathmore University  
**Date:** June 2026  

---

### Dataset Source
Kenya National Bureau of Statistics (KNBS) — *Kenya Housing Survey 2023/24*  
Portal: https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

---

> **Scope of this notebook:** Data loading → variable inventory → preprocessing → descriptive statistics → graphical EDA → distributional assessment.  
> Parametric and non-parametric inference follow in a separate notebook.


---
## 0. Environment Setup


In [1]:
# ── 0.1  Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


Mounted at /content/drive
Drive mounted.


In [2]:
# ── 0.2  Install dependencies (first run only) ───────────────────────────
!pip install -q pyreadstat polars pyarrow
print('Dependencies ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 41.2 MB/s eta 0:00:00
Dependencies ready.


In [3]:
# ── 0.3  Core imports ────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, probplot, norm as spnorm
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})

TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'
DARK   = '#2C2C2A'; GREEN  = '#2E7D32'

print('All imports loaded.')


All imports loaded.


In [4]:
# ── 0.4  Paths and county map ────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
RAW   = DRIVE / 'data' / 'raw'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'Paths ready.  FIGS={FIGS}  TABS={TABS}')


Paths ready.  FIGS=/content/drive/MyDrive/KHS_Dissertation/outputs/figures/dsa8301  TABS=/content/drive/MyDrive/KHS_Dissertation/outputs/tables/dsa8301


---
## 1. Dataset Description

### 1.1 Source & Background

The **Kenya Housing Survey (KHS) 2023/24** is a nationally representative household survey conducted by KNBS. It covers **21,347 households** across all **47 counties**, collecting data on housing conditions, tenure, infrastructure, household finances, and demographic composition.

The survey ships as six Stata (.dta) files, converted here to Parquet for efficiency:

| File key | Unit of observation | Core content |
|----------|---------------------|--------------|
| `household` | Household (spine) | Finances, tenure, utilities, infrastructure — 392 columns |
| `individual` | Person | Demographics, education, employment |
| `dwelling` | Dwelling unit | Wall/roof/floor materials, rooms, floor area |
| `land_parcels` | Land parcel | Tenure system, title documents, eviction risk |
| `county` | County (47 rows) | Physical planning, infrastructure indicators |
| `mortgage` | Mortgage record | Demand, products, county-level coverage |


In [5]:
# ── 1.2  Load all parquet files ─────────────────────────────────────────
FILES = {
    'household'   : 'Household_Information_Data.parquet',
    'individual'  : 'Individual_Data.parquet',
    'dwelling'    : 'Dwelling_Units_Data.parquet',
    'land_parcels': 'Land_Parcels_Data.parquet',
    'county'      : 'County_Physical_Planning_Data.parquet',
    'mortgage'    : 'Housing_Mortgage_Data.parquet',


    'loan'         : 'Housing_Loans_Data.parquet',
    # New — previously unused
    'nema'         : 'NEMA_Data_Set.parquet',
    'water_svc'    : 'Water_Services_Providers_Data.parquet',
    'real_estate'  : 'Real_Estate_Dataset.parquet',
    'financiers'   : 'Housing_Financiers_Data.parquet',
    'institutional': 'KHS_Institutional_Data.parquet',
    'project_info' : 'Project Information.parquet',
    'housing_types': 'Type of Housing Units.parquet',



}

dfs = {}
print(f'  {"File":<15} {"Rows":>8} {"Cols":>6}')
print('  ' + '-'*32)
for key, fname in FILES.items():
    path = PQ / fname
    if not path.exists():
        print(f'  {key:<15} NOT FOUND')
        continue
    df = pd.read_parquet(path)
    dfs[key] = df
    print(f'  {key:<15} {df.shape[0]:>8,} {df.shape[1]:>6}')

hh  = dfs.get('household')
ind = dfs.get('individual')
dw  = dfs.get('dwelling')
print(f'\nPrimary frame: household  =>  {hh.shape[0]:,} rows x {hh.shape[1]:,} cols')


  File                Rows   Cols
  --------------------------------
  household         21,347    392
  individual        80,889     97
  dwelling          25,116     25
  land_parcels      11,136     34
  county                47    116
  mortgage           1,644     13
  loan                 946     10
  nema                  48     45
  water_svc            153     96
  real_estate        7,236    300
  financiers           351     63
  institutional        348    194
  project_info          71    211
  housing_types        131     17

Primary frame: household  =>  21,347 rows x 392 cols


In [6]:
# ── 1.3  Variable Registry ──────────────────────────────────────────────
# A reference map of analysis-relevant variables across all loaded files.
# type: continuous | ordinal | binary | categorical

VARIABLE_REGISTRY = {

    # ── IDENTIFIERS & WEIGHTS ────────────────────────────────────────────
    'interview__key' : {'label': 'Household unique interview key',                   'type': 'id',          'file': 'household'},
    'a01'            : {'label': 'County code (1–47)',                               'type': 'categorical', 'file': 'household'},
    'countycode'     : {'label': 'County code string-padded (01–47)',                'type': 'categorical', 'file': 'household'},
    'a07_1'          : {'label': 'Urban/Rural stratum (1=Rural, 2=Urban)',           'type': 'binary',      'file': 'household'},
    'serial'         : {'label': 'KNBS household serial number',                     'type': 'id',          'file': 'household'},
    'hhweight'       : {'label': 'Household survey weight',                          'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — WATER & SANITATION ───────────────────────────────────
    'c01_1'          : {'label': 'Main drinking water source (1=piped HH, 4=borehole, 10=river)', 'type': 'ordinal',     'file': 'household'},
    'c01_2'          : {'label': 'Water collection method (1=piped, 5=fetched)',     'type': 'ordinal',     'file': 'household'},
    'c01_3'          : {'label': 'Water treated before drinking (0=No, 1=Yes)',      'type': 'binary',      'file': 'household'},
    'c01_4'          : {'label': 'Time to water source (minutes, one way)',          'type': 'continuous',  'file': 'household'},
    'c02_1'          : {'label': 'Secondary drinking water source',                  'type': 'ordinal',     'file': 'household'},
    'c04'            : {'label': 'Main toilet facility type (1=flush, 7=pit, 8=none)', 'type': 'ordinal',   'file': 'household'},
    'c05'            : {'label': 'Handwashing facility available (0=No, 1=Yes)',     'type': 'binary',      'file': 'household'},
    'c07'            : {'label': 'Handwashing materials present (4=soap+water)',     'type': 'ordinal',     'file': 'household'},
    'c14_1'          : {'label': 'Monthly water expenditure (KES)',                  'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — ENERGY ───────────────────────────────────────────────
    'c10'            : {'label': 'Main lighting source (1=grid electricity, 4=solar, 5=kerosene)', 'type': 'ordinal', 'file': 'household'},
    'c10_2'          : {'label': 'Hours of electricity supply per day',              'type': 'continuous',  'file': 'household'},
    'c10_4'          : {'label': 'Electricity connection type (0=prepaid, 1=postpaid)', 'type': 'binary',   'file': 'household'},
    'c11'            : {'label': 'Main cooking fuel (7=firewood, 9=charcoal, 11=LPG)', 'type': 'ordinal',  'file': 'household'},
    'c12'            : {'label': 'Primary cooking stove type (1=3-stone, 6=improved, 10=gas)', 'type': 'ordinal', 'file': 'household'},
    'c14_2'          : {'label': 'Monthly electricity expenditure (KES)',            'type': 'continuous',  'file': 'household'},
    'c14_3'          : {'label': 'Monthly other energy expenditure (KES)',           'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — ASSETS ───────────────────────────────────────────────
    'c13__1'         : {'label': 'Owns radio (0=No, 1=Yes)',                         'type': 'binary',      'file': 'household'},
    'c13__2'         : {'label': 'Owns mobile phone (0=No, 1=Yes)',                  'type': 'binary',      'file': 'household'},
    'c13__3'         : {'label': 'Owns television (0=No, 1=Yes)',                    'type': 'binary',      'file': 'household'},
    'c13__4'         : {'label': 'Owns computer/laptop (0=No, 1=Yes)',               'type': 'binary',      'file': 'household'},
    'c13__5'         : {'label': 'Owns motorcycle (0=No, 1=Yes)',                    'type': 'binary',      'file': 'household'},
    'c13__6'         : {'label': 'Owns motor vehicle (0=No, 1=Yes)',                 'type': 'binary',      'file': 'household'},
    'c13__7'         : {'label': 'Owns refrigerator (0=No, 1=Yes)',                  'type': 'binary',      'file': 'household'},
    'internet'       : {'label': 'Household has internet access (1=Yes)',            'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — INCOME & EXPENDITURE ─────────────────────────────────
    'g01a'           : {'label': 'Monthly food expenditure (KES)',                   'type': 'continuous',  'file': 'household'},
    'g01b'           : {'label': 'Monthly clothing expenditure (KES)',               'type': 'continuous',  'file': 'household'},
    'g01c'           : {'label': 'Monthly education expenditure (KES)',              'type': 'continuous',  'file': 'household'},
    'g01d'           : {'label': 'Monthly health expenditure (KES)',                 'type': 'continuous',  'file': 'household'},
    'g01e'           : {'label': 'Monthly transport expenditure (KES)',              'type': 'continuous',  'file': 'household'},
    'g01f'           : {'label': 'Monthly communication expenditure (KES)',          'type': 'continuous',  'file': 'household'},
    'g01g'           : {'label': 'Monthly recreation expenditure (KES)',             'type': 'continuous',  'file': 'household'},
    'g01h'           : {'label': 'Monthly housing cost expenditure (KES)',           'type': 'continuous',  'file': 'household'},
    'g01i'           : {'label': 'Monthly energy expenditure (KES)',                 'type': 'continuous',  'file': 'household'},
    'g01j'           : {'label': 'Monthly other expenditure (KES)',                  'type': 'continuous',  'file': 'household'},
    'g01k'           : {'label': 'Monthly remittances sent (KES)',                   'type': 'continuous',  'file': 'household'},
    'g02'            : {'label': 'Household pays rent (1=Yes, 2=No)',                'type': 'binary',      'file': 'household'},
    'g02_1'          : {'label': 'Monthly rent paid (KES)',                          'type': 'continuous',  'file': 'household'},
    'g03'            : {'label': 'Housing tenure type (1=owner, 2=tenant, 3=other)', 'type': 'categorical', 'file': 'household'},
    'g04'            : {'label': 'Owns other property (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — HOUSING PROBLEMS & ASPIRATIONS ───────────────────────
    'g05__1'         : {'label': 'Problem: overcrowding (0=No, 1=Yes)',              'type': 'binary',      'file': 'household'},
    'g05__2'         : {'label': 'Problem: poor water supply (0=No, 1=Yes)',         'type': 'binary',      'file': 'household'},
    'g05__3'         : {'label': 'Problem: poor sanitation (0=No, 1=Yes)',           'type': 'binary',      'file': 'household'},
    'g05__4'         : {'label': 'Problem: poor drainage (0=No, 1=Yes)',             'type': 'binary',      'file': 'household'},
    'g05__5'         : {'label': 'Problem: poor road access (0=No, 1=Yes)',          'type': 'binary',      'file': 'household'},
    'g05__6'         : {'label': 'Problem: insecurity (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},
    'g05__7'         : {'label': 'Problem: high rent/housing cost (0=No, 1=Yes)',    'type': 'binary',      'file': 'household'},
    'g05__8'         : {'label': 'Problem: poor structural condition (0=No, 1=Yes)', 'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — HOUSING PERCEPTION (h01–h11) ─────────────────────────
    'h01'            : {'label': 'Perceived structural quality (1=Good, 2=Fair, 3=Poor)',    'type': 'ordinal', 'file': 'household'},
    'h02'            : {'label': 'Perceived roof quality (1=Good, 2=Fair, 3=Poor)',          'type': 'ordinal', 'file': 'household'},
    'h03'            : {'label': 'Perceived wall quality (1=Good, 2=Fair, 3=Poor)',          'type': 'ordinal', 'file': 'household'},
    'h04'            : {'label': 'Perceived floor quality (1=Good, 2=Fair, 3=Poor)',         'type': 'ordinal', 'file': 'household'},
    'h05'            : {'label': 'Perceived ventilation adequacy (1=Good, 2=Fair, 3=Poor)',  'type': 'ordinal', 'file': 'household'},
    'h06'            : {'label': 'Perceived natural lighting (1=Good, 2=Fair, 3=Poor)',      'type': 'ordinal', 'file': 'household'},
    'h07'            : {'label': 'Perceived water supply adequacy (1=Good, 2=Fair, 3=Poor)', 'type': 'ordinal', 'file': 'household'},
    'h08'            : {'label': 'Perceived sanitation adequacy (1=Good, 2=Fair, 3=Poor)',   'type': 'ordinal', 'file': 'household'},
    'h09'            : {'label': 'Perceived waste disposal (1=Good, 2=Fair, 3=Poor)',        'type': 'ordinal', 'file': 'household'},
    'h10'            : {'label': 'Perceived neighbourhood security (1=Good, 2=Fair, 3=Poor)','type': 'ordinal', 'file': 'household'},
    'h11'            : {'label': 'Overall housing satisfaction (1=Good, 2=Fair, 3=Poor)',    'type': 'ordinal', 'file': 'household'},

    # ── HOUSEHOLD — TENURE & MOBILITY (j-module) ─────────────────────────
    'j04_1'          : {'label': 'Current tenure arrangement (1=owner, 0=other)',    'type': 'binary',      'file': 'household'},
    'j05'            : {'label': 'Has formal title/ownership document (0=No, 1=Yes)','type': 'binary',      'file': 'household'},
    'j09'            : {'label': 'Housing cost is a financial burden (0=No, 1=Yes)', 'type': 'binary',      'file': 'household'},
    'j10'            : {'label': 'Ever missed rent/mortgage payment (0=No, 1=Yes)',  'type': 'binary',      'file': 'household'},
    'j11'            : {'label': 'At risk of eviction in next 12 months (0=No, 1=Yes)', 'type': 'binary',   'file': 'household'},
    'j12_1'          : {'label': 'Years in current dwelling (1=<1yr, coded year otherwise)', 'type': 'ordinal', 'file': 'household'},
    'j13'            : {'label': 'Satisfied with current tenure (0=No, 1=Yes)',      'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — RENTAL MODULE (k-module, renters only ~32%) ──────────
    'k02'            : {'label': 'Written tenancy agreement exists (1=Yes, 2=No)',   'type': 'binary',      'file': 'household'},
    'k05'            : {'label': 'Monthly rent (KES)',                               'type': 'continuous',  'file': 'household'},
    'k09'            : {'label': 'Lease type (1=monthly, 2=annual)',                 'type': 'categorical', 'file': 'household'},
    'k21'            : {'label': 'Rent arrears status (1=in arrears, others)',       'type': 'ordinal',     'file': 'household'},
    'min_rent'       : {'label': 'Minimum rent in PSU — local market floor (KES)',   'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — OWNED DWELLING MODULE (l-module) ─────────────────────
    'l07'            : {'label': 'Year dwelling was built',                          'type': 'continuous',  'file': 'household'},
    'l13'            : {'label': 'Monthly mortgage/housing loan repayment (KES)',    'type': 'continuous',  'file': 'household'},
    'l14'            : {'label': 'Estimated market value of dwelling (KES)',         'type': 'continuous',  'file': 'household'},
    'l15'            : {'label': 'Imputed monthly housing cost/rent equivalent (KES)', 'type': 'continuous','file': 'household'},
    'l19'            : {'label': 'Plot/land size (decimal: 12=1 acre)',              'type': 'continuous',  'file': 'household'},
    'l21'            : {'label': 'Major renovation done (0=No, 1=Yes)',              'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — ENVIRONMENT & HAZARDS (e-module) ─────────────────────
    'e01'            : {'label': 'Solid waste disposal method (0=collected, 9=open dump)', 'type': 'ordinal', 'file': 'household'},
    'e05'            : {'label': 'Proximity to waste dump/quarry (0=No, 1=Yes)',     'type': 'binary',      'file': 'household'},
    'e06'            : {'label': 'Flood exposure (0=none, 1=severe, 2=mild)',        'type': 'ordinal',     'file': 'household'},
    'e07'            : {'label': 'Mudslide/erosion exposure (0=none, 1=severe, 2=mild)', 'type': 'ordinal', 'file': 'household'},
    'e08'            : {'label': 'Terrain/slope type (1=flat, 2=gentle, 3=hilly, 4=steep)', 'type': 'ordinal', 'file': 'household'},

    # ── HOUSEHOLD — LAND OWNERSHIP (i00) ─────────────────────────────────
    'i00'            : {'label': 'Household owns land (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — DERIVED / COMPUTED ───────────────────────────────────
    'prop_util'      : {'label': 'Utilities as proportion of income (ratio)',        'type': 'continuous',  'file': 'household'},
    'med_prop'       : {'label': 'Median utility-to-income ratio, county level',     'type': 'continuous',  'file': 'household'},
    'utilities'      : {'label': 'Household pays for utilities (0=No, 1=Yes)',       'type': 'binary',      'file': 'household'},
    'ctymin_ut'      : {'label': 'County-level minimum utility cost (KES)',          'type': 'continuous',  'file': 'household'},
    'med_brms'       : {'label': 'Median bedrooms in county (rooms)',                'type': 'continuous',  'file': 'household'},
    'sf'             : {'label': 'Slum/informal settlement flag (1=slum, 2=non-slum)', 'type': 'binary',    'file': 'household'},
    'pln'            : {'label': 'Planning status of settlement (coded)',            'type': 'categorical', 'file': 'household'},

    # ── DWELLING FILE ─────────────────────────────────────────────────────
    'd01'            : {'label': 'Dwelling tenure type (1=owner-occupied, 2=rented, 3=other)', 'type': 'categorical', 'file': 'dwelling'},
    'd03'            : {'label': 'Dwelling type (1=conventional house, 4=flat, 7=traditional)', 'type': 'categorical', 'file': 'dwelling'},
    'd05'            : {'label': 'Located in approved building (1=Yes, 0=No)',       'type': 'binary',      'file': 'dwelling'},
    'd06'            : {'label': 'Building has planning approval (1=Yes, 0=No)',     'type': 'binary',      'file': 'dwelling'},
    'd07'            : {'label': 'Dwelling in hazard-prone area (1=Yes, 0=No)',      'type': 'binary',      'file': 'dwelling'},
    'd08'            : {'label': 'Outer wall material (1=stone/brick, 3=timber, 5=mud/earth)', 'type': 'ordinal', 'file': 'dwelling'},
    'd09'            : {'label': 'Roof material (1=iron sheets, 2=tiles, 3=grass/thatch)', 'type': 'ordinal', 'file': 'dwelling'},
    'd10'            : {'label': 'Floor material (1=cement, 2=tiles, 3=earth)',      'type': 'ordinal',     'file': 'dwelling'},
    'd11'            : {'label': 'Number of rooms in dwelling',                      'type': 'continuous',  'file': 'dwelling'},
    'd11_1'          : {'label': 'Floor area of dwelling (sq m)',                    'type': 'continuous',  'file': 'dwelling'},
    'd11_2'          : {'label': 'Number of bedrooms',                              'type': 'continuous',  'file': 'dwelling'},
    'd12'            : {'label': 'Number of rooms used for sleeping',                'type': 'continuous',  'file': 'dwelling'},

    # ── INDIVIDUAL FILE ───────────────────────────────────────────────────
    'b04'            : {'label': 'Sex of individual (1=Male, 2=Female)',             'type': 'binary',      'file': 'individual'},
    'b05_years'      : {'label': 'Age in completed years',                           'type': 'continuous',  'file': 'individual'},
    'b07'            : {'label': 'Marital status (1=never married, 2=married, 3=divorced, 4=widowed)', 'type': 'categorical', 'file': 'individual'},
    'b10'            : {'label': 'Currently attending school (0=No, 1=Yes)',         'type': 'binary',      'file': 'individual'},
    'b11'            : {'label': 'Literacy status (0=illiterate, 1=literate)',       'type': 'binary',      'file': 'individual'},
    'ken_edu_isced11': {'label': 'Highest education level (ISCED-11: 3=primary, 6=secondary, 16=tertiary)', 'type': 'ordinal', 'file': 'individual'},
    'any_disability' : {'label': 'Any functional disability (0=No, 1=Yes)',          'type': 'binary',      'file': 'individual'},
    'resid'          : {'label': 'Residence type (1=Urban, 2=Rural)',                'type': 'binary',      'file': 'individual'},
    'hhsize'         : {'label': 'Total persons in household',                       'type': 'continuous',  'file': 'individual'},
    'age_dep'        : {'label': 'Age dependency status (0=child <15, 15=working age, 65=elderly)', 'type': 'ordinal', 'file': 'individual'},
    'wap'            : {'label': 'Working-age population flag (1=WAP, NaN=not)',     'type': 'binary',      'file': 'individual'},
    'inw'            : {'label': 'Individual survey weight',                         'type': 'continuous',  'file': 'individual'},

    # ── LAND PARCELS FILE ─────────────────────────────────────────────────
    'i01_3'          : {'label': 'Land ownership type (1=freehold, 2=leasehold, 3=customary)', 'type': 'categorical', 'file': 'land_parcels'},
    'i05'            : {'label': 'Land has title deed (1=Yes, 3=no, 14=other)',      'type': 'ordinal',     'file': 'land_parcels'},
    'i06'            : {'label': 'Land use type (1=residential, 12=agricultural, 14=mixed)', 'type': 'categorical', 'file': 'land_parcels'},
    'i08'            : {'label': 'Land dispute in last 5 years (0=No, 1=Yes)',       'type': 'binary',      'file': 'land_parcels'},
    'i10'            : {'label': 'Land registered (0=No, 1=Yes)',                   'type': 'binary',      'file': 'land_parcels'},
    'i12'            : {'label': 'Land used as loan collateral (0=No, 1=Yes)',       'type': 'binary',      'file': 'land_parcels'},

}

# ── Summary ──────────────────────────────────────────────────────────────
_reg_df = pd.DataFrame(VARIABLE_REGISTRY).T
print(f'  Total variables registered: {len(VARIABLE_REGISTRY)}')
print()
print(_reg_df.groupby(['file', 'type']).size().rename('count').to_string())

  Total variables registered: 125

file          type       
dwelling      binary          3
              categorical     2
              continuous      4
              ordinal         3
household     binary         34
              categorical     5
              continuous     29
              id              2
              ordinal        25
individual    binary          6
              categorical     1
              continuous      3
              ordinal         2
land_parcels  binary          3
              categorical     2
              ordinal         1


In [7]:
# ── 1.4  Join feasibility ────────────────────────────────────────────────
# For every non-household file, test whether interview__key links back
# to the household spine.  Results inform the merge strategy in 1.7.

SPINE_KEY   = 'interview__key'
hh_keys     = set(hh[SPINE_KEY].dropna())
hh_counties = set(hh['a01'].dropna().astype(int))

print("JOIN FEASIBILITY MATRIX")
print("=" * 70)
print(f"Household spine : {len(hh_keys):,} unique interview__key values\n")
print(f"  {'File':<18} {'Rows':>7}  {'HH-level join':>22}  {'County col'}")
print("  " + "─" * 68)

for key, df in dfs.items():
    if key == 'household':
        continue

    n_rows = f"{df.shape[0]:,}"

    # household-level match
    hh_match = ''
    if SPINE_KEY in df.columns:
        file_keys  = set(df[SPINE_KEY].dropna())
        overlap    = len(file_keys & hh_keys)
        pct        = overlap / len(hh_keys) * 100
        hh_match   = f"{overlap:,} / {len(hh_keys):,}  ({pct:.1f}%)"
    else:
        hh_match   = "no interview__key"

    # county bridge column
    cty_col = next(
        (c for c in df.columns if c.lower() in
         ('a01', 'cg00', 'nm00', 'county_name', 'county_code', 'countycode')),
        '—'
    )

    print(f"  {key:<18} {n_rows:>7}  {hh_match:>22}  {cty_col}")

print()
print("  Conclusion")
print("  ─" * 35)
print("  100% match  → direct left join on interview__key")
print("  45.5% match → left join (NaN = household owns no land — see 1.5)")
print("  0% match    → aggregate to county then join via a01")



JOIN FEASIBILITY MATRIX
Household spine : 21,347 unique interview__key values

  File                  Rows           HH-level join  County col
  ────────────────────────────────────────────────────────────────────
  individual          80,889  21,346 / 21,347  (100.0%)  a01
  dwelling            25,116  21,346 / 21,347  (100.0%)  a01
  land_parcels        11,136  9,707 / 21,347  (45.5%)  —
  county                  47      0 / 21,347  (0.0%)  cg00
  mortgage             1,644      0 / 21,347  (0.0%)  county_name
  loan                   946      0 / 21,347  (0.0%)  county_name
  nema                    48      0 / 21,347  (0.0%)  nm00
  water_svc              153      0 / 21,347  (0.0%)  county_name
  real_estate          7,236       no interview__key  county_name
  financiers             351      0 / 21,347  (0.0%)  county_name
  institutional          348      0 / 21,347  (0.0%)  county_name
  project_info            71      0 / 21,347  (0.0%)  —
  housing_types          131      0 

In [8]:


# ── 1.5  Land parcels — structural coverage check ────────────────────────
# 45.5% HH coverage: hypothesis is that parcel records exist iff i00 == 1.

print("\n" + "=" * 70)
print("LAND_PARCELS — is the 45.5% gap explained by i00 (land ownership)?")
print("=" * 70)

lp      = dfs['land_parcels'].copy()
lp_keys = set(lp[SPINE_KEY].dropna())

hh_sub = hh[['interview__key', 'i00']].copy()
hh_sub['has_parcel'] = hh_sub['interview__key'].isin(lp_keys).astype(int)

print(f"\n  Parcel records          : {len(lp):,}")
print(f"  HHs with  parcel record : {len(lp_keys & hh_keys):,}"
      f"  ({len(lp_keys & hh_keys)/len(hh_keys)*100:.1f}%)")
print(f"  HHs without parcel      : {len(hh_keys - lp_keys):,}"
      f"  ({len(hh_keys - lp_keys)/len(hh_keys)*100:.1f}%)")

print(f"\n  i00 × parcel presence:")
print(f"  {'i00':<6} {'Has parcel':>12} {'No parcel':>12} {'% with parcel':>15}")
print("  " + "─" * 48)
for val in sorted(hh_sub['i00'].dropna().unique()):
    sub = hh_sub[hh_sub['i00'] == val]
    has = sub['has_parcel'].sum()
    no  = len(sub) - has
    pct = has / len(sub) * 100
    print(f"  {val:<6.0f} {has:>12,} {no:>12,} {pct:>14.1f}%")

sub_nan = hh_sub[hh_sub['i00'].isna()]
if len(sub_nan):
    has = sub_nan['has_parcel'].sum()
    no  = len(sub_nan) - has
    pct = has / len(sub_nan) * 100 if len(sub_nan) else 0
    print(f"  {'NaN':<6} {has:>12,} {no:>12,} {pct:>14.1f}%")

pph = lp.groupby(SPINE_KEY).size().value_counts().sort_index().to_dict()
print(f"\n  Parcels per household : {pph}")
print("\n  ✓ Structurally complete: parcel record exists iff i00 = 1.")
print("    Left join is correct — NaN means no land owned, not missing data.")



LAND_PARCELS — is the 45.5% gap explained by i00 (land ownership)?

  Parcel records          : 11,136
  HHs with  parcel record : 9,707  (45.5%)
  HHs without parcel      : 11,640  (54.5%)

  i00 × parcel presence:
  i00      Has parcel    No parcel   % with parcel
  ────────────────────────────────────────────────
  0                 0       11,639            0.0%
  1             9,707            0          100.0%
  NaN               0            1            0.0%

  Parcels per household : {1: 8520, 2: 1004, 3: 147, 4: 30, 5: 7, 6: 2}

  ✓ Structurally complete: parcel record exists iff i00 = 1.
    Left join is correct — NaN means no land owned, not missing data.


In [9]:


# ── 1.6  County-level aggregates ──────────────────────────────────────────
# Files with 0% HH-level match are aggregated to 47 county summaries,
# then joined to the master frame via a01 (county code integer).

import unicodedata, re, numpy as np

# County name → integer code lookup (covers messy strings in the data)
def _norm(s):
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r"['\-]", '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

NAME_TO_CODE = {_norm(v): k for k, v in COUNTY_MAP.items()}
NAME_TO_CODE.update({
    'nairobi city'  : 47, 'nairobi'         : 47,
    'muranga'       : 21, 'taita taveta'     : 6,
    'tharaka nithi' : 13, 'elgeyo marakwet'  : 28,
    'uasin gishu'   : 27, 'homa bay'         : 43,
    'homabay'       : 43,
})

def to_county_code(series):
    return series.map(lambda x: NAME_TO_CODE.get(_norm(str(x)), pd.NA))

# ── COUNTY ───────────────────────────────────────────────────────────────
county_df = dfs['county'].copy()
county_df['a01'] = (county_df['cg00'].astype(str).str.strip()
                                      .str.lstrip('0')
                                      .replace('', pd.NA)
                                      .astype(float)
                                      .astype('Int64'))
county_agg = county_df[['a01', 'cg1a', 'cg1b', 'cg3', 'cg11', 'cg12']].rename(columns={
    'cg1a' : 'cty_housing_stock',
    'cg1b' : 'cty_housing_backlog',
    'cg3'  : 'cty_planning_staff',
    'cg11' : 'cty_has_housing_policy',
    'cg12' : 'cty_building_approval_system',
})

# ── NEMA ─────────────────────────────────────────────────────────────────
nema_df = dfs['nema'].copy()
nema_df['a01'] = nema_df['nm00'].astype('Int64')
nema_agg = nema_df[['a01', 'nema1a', 'nema1b', 'nema6']].rename(columns={
    'nema1a' : 'nema_eia_applications',
    'nema1b' : 'nema_eia_approvals',
    'nema6'  : 'nema_processing_days',
})

# ── WATER_SVC ────────────────────────────────────────────────────────────
# county_name stores integer county codes in this file
wsvc_df = dfs['water_svc'].copy()
wsvc_df['a01'] = pd.to_numeric(wsvc_df['county_name'], errors='coerce').astype('Int64')
wsvc_agg = (
    wsvc_df[['a01', 'wssp1a', 'wssp1b', 'wssp7', 'wssp12']]
    .rename(columns={
        'wssp1a'  : 'wsvc_water_connections',
        'wssp1b'  : 'wsvc_sewer_connections',
        'wssp7'   : 'wsvc_water_tariff',
        'wssp12'  : 'wsvc_service_quality',
    })
    .groupby('a01', as_index=False).agg(
        wsvc_water_connections = ('wsvc_water_connections', 'sum'),
        wsvc_sewer_connections = ('wsvc_sewer_connections', 'sum'),
        wsvc_water_tariff      = ('wsvc_water_tariff',      'mean'),
        wsvc_service_quality   = ('wsvc_service_quality',   'mean'),
        wsvc_n_providers       = ('wsvc_water_tariff',      'count'),
    )
)

# ── MORTGAGE ─────────────────────────────────────────────────────────────
mort_df = dfs['mortgage'].copy()
mort_df['a01'] = to_county_code(mort_df['county_name'])
mort_agg = (
    mort_df[['a01', 'se6b', 'se8a', 'se9b']]
    .rename(columns={
        'se6b' : 'mort_interest_rate',
        'se8a' : 'mort_ltv_ratio',
        'se9b' : 'mort_avg_term_years',
    })
    .groupby('a01', as_index=False).agg(
        mort_interest_rate  = ('mort_interest_rate',  'mean'),
        mort_ltv_ratio      = ('mort_ltv_ratio',       'mean'),
        mort_avg_term_years = ('mort_avg_term_years',  'mean'),
        mort_n_providers    = ('mort_interest_rate',   'count'),
    )
)

# ── LOAN ─────────────────────────────────────────────────────────────────
loan_df = dfs['loan'].copy()
loan_df['a01'] = to_county_code(loan_df['county_name'])
loan_agg = (
    loan_df[['a01', 'se3c', 'se3d']]
    .rename(columns={
        'se3c' : 'loan_avg_size',
        'se3d' : 'loan_outstanding',
    })
    .groupby('a01', as_index=False).agg(
        loan_avg_size    = ('loan_avg_size',    'mean'),
        loan_outstanding = ('loan_outstanding', 'mean'),
        loan_n_providers = ('loan_avg_size',    'count'),
    )
)

# ── FINANCIERS ────────────────────────────────────────────────────────────
fin_df = dfs['financiers'].copy()
fin_df['a01'] = to_county_code(fin_df['county_name'])
fin_agg = (
    fin_df[['a01', 'se4a', 'se7']]
    .rename(columns={
        'se4a' : 'fin_portfolio',
        'se7'  : 'fin_avg_tenure_months',
    })
    .groupby('a01', as_index=False).agg(
        fin_portfolio          = ('fin_portfolio',          'sum'),
        fin_avg_tenure_months  = ('fin_avg_tenure_months',  'mean'),
        fin_n_financiers       = ('fin_portfolio',          'count'),
    )
)

print("County aggregates built:")
for name, df_ in [
    ('county_agg', county_agg), ('nema_agg',  nema_agg),
    ('wsvc_agg',   wsvc_agg),   ('mort_agg',  mort_agg),
    ('loan_agg',   loan_agg),   ('fin_agg',   fin_agg),
]:
    print(f"  {name:<14} {df_.shape[0]:>3} rows × {df_.shape[1]:>2} cols  "
          f"| a01 coverage: {df_['a01'].notna().sum()} / 47 counties")



County aggregates built:
  county_agg      47 rows ×  6 cols  | a01 coverage: 47 / 47 counties
  nema_agg        48 rows ×  4 cols  | a01 coverage: 48 / 47 counties
  wsvc_agg        45 rows ×  6 cols  | a01 coverage: 45 / 47 counties
  mort_agg        38 rows ×  5 cols  | a01 coverage: 38 / 47 counties
  loan_agg        38 rows ×  4 cols  | a01 coverage: 38 / 47 counties
  fin_agg         38 rows ×  4 cols  | a01 coverage: 38 / 47 counties


In [10]:


# ── 1.6b  Real Estate / Institutional / Project aggregates (NEW — previously
#          loaded into `dfs` but never merged into master) ─────────────────
# Same pattern as 1.6: these are establishment-level surveys, aggregated to
# the 47 counties and left-joined to every household in that county.

# ── REAL ESTATE (developers / agencies) ────────────────────────────────────
re_df = dfs['real_estate'].copy()
re_df['a01'] = to_county_code(re_df['county_name'])
# One row per establishment first — ra/rb/rc/rd/re sections repeat within an
# establishment (residential/commercial/warehousing blocks), so aggregating
# the raw 7,236 rows directly would double-count establishments.
re_est = re_df.sort_values('interview__id').drop_duplicates('interview__id')
realest_agg = (
    re_est.groupby('a01', as_index=False)
    .agg(realest_n_establishments=('interview__id', 'count'))
)
# NOTE: the file has ~300 columns across residential/commercial/warehousing
# blocks (ra7 etc. are multi-select 0/1 flags, not counts) — only the
# establishment count is aggregated here since a mean of a binary flag would
# be a proportion, not a meaningful "avg units" figure. Extend this block
# with specific ra/rb/rc/rd fields once you've picked which ones you need
# and confirmed their coding against the questionnaire.
print(f"realest_agg   : {realest_agg.shape[0]} counties  "
      f"(raw {len(re_df)} rows, {len(re_est)} unique establishments, "
      f"{re_est['a01'].notna().sum()}/{len(re_est)} with a resolvable county)")

# ── INSTITUTIONAL (county government / physical planning offices) ─────────
# Column names not independently verified against this file in this
# session — auto-detect the county key defensively instead of guessing.
inst_df = dfs['institutional'].copy()
_county_key_candidates = ['a01', 'county_name', 'countycode', 'cg00', 'county_code']
_inst_county_col = next((c for c in _county_key_candidates if c in inst_df.columns), None)

if _inst_county_col is None:
    print(f"⚠  institutional: no recognizable county key among {_county_key_candidates}.")
    print(f"   Columns available: {list(inst_df.columns)[:20]} ...")
    print("   Skipping institutional merge — add the correct key above once identified.")
    institutional_agg = pd.DataFrame({'a01': pd.Series(dtype='Int64')})
else:
    if pd.api.types.is_numeric_dtype(inst_df[_inst_county_col]):
        inst_df['a01'] = pd.to_numeric(inst_df[_inst_county_col], errors='coerce').astype('Int64')
    else:
        inst_df['a01'] = to_county_code(inst_df[_inst_county_col])
    institutional_agg = (
        inst_df.groupby('a01', as_index=False)
        .agg(inst_n_establishments=(_inst_county_col, 'count'))
    )
    print(f"institutional_agg : {institutional_agg.shape[0]} counties  "
          f"(key='{_inst_county_col}', raw {len(inst_df)} rows)")
    print(f"   Only a row-count placeholder is built — inspect inst_df.columns and extend "
          f"with the specific fields you want (e.g. approvals, staffing) before relying on this.")

# ── PROJECT_INFO + HOUSING_TYPES ────────────────────────────────────────────
# Confirmed: Project_Information and Type_of_Housing_Units carry NO county
# or household key in the raw data (checked directly against both files) —
# they nest as establishment -> project (c3_2__id) -> unit type (c3_32r__id)
# but never touch a01/county_name. They cannot be merged into master at
# household or county level. Reported as a standalone national summary
# instead of silently dropped.
proj_df = dfs['project_info'].copy()
htype_df = dfs['housing_types'].copy()

_proj_county_col = next((c for c in _county_key_candidates if c in proj_df.columns), None)
if _proj_county_col:
    print(f"project_info: found county key '{_proj_county_col}' — extend this block to merge by county.")
else:
    htype_units = htype_df.groupby('interview__key', as_index=False).agg(
        proj_units_total=('Total_Units_Check', 'sum') if 'Total_Units_Check' in htype_df.columns
                          else ('c3_32r__id', 'count'),
        proj_unit_types_n=('c3_32r__id', 'count'),
    )
    proj_summary = (
        proj_df.groupby('interview__key', as_index=False)
        .agg(proj_n_projects=('c3_2__id', 'count'))
        .merge(htype_units, on='interview__key', how='left')
    )
    proj_summary.to_csv(PQ.parent / 'project_information_summary.csv', index=False)
    print(f"project_info + housing_types: no county/household key present — saved standalone "
          f"national summary ({len(proj_summary)} establishments) to "
          f"project_information_summary.csv, NOT merged into master.")


realest_agg   : 14 counties  (raw 7236 rows, 272 unique establishments, 272/272 with a resolvable county)
institutional_agg : 16 counties  (key='county_name', raw 348 rows)
   Only a row-count placeholder is built — inspect inst_df.columns and extend with the specific fields you want (e.g. approvals, staffing) before relying on this.
project_info + housing_types: no county/household key present — saved standalone national summary (32 establishments) to project_information_summary.csv, NOT merged into master.


In [11]:
# ── 1.7  Individual → household aggregates (FIXED) ───────────────────────
# Fix 1: hh_size = actual number of individual records per household, not
#        the self-reported roster length (b02_length), which disagrees for
#        4 households (e.g. roster says 9 members, only 8 records exist).
#        Every other individual-level aggregate below is computed from the
#        actual rows, so hh_size has to match that denominator or the
#        dependency ratio / per-capita figures for those 4 rows will be
#        quietly wrong.
# Fix 2: hh_head_sex derived from relationship_details==1 (HEAD, per the
#        B02 codebook) joined to sex, instead of the raw 'hhh_sex' column
#        which is NaN for 73.7% of rows.

ind = dfs['individual'].copy()

actual_hh_size   = ind.groupby('interview__key').size().rename('hh_size')
reported_hh_size = ind.groupby('interview__key')['b02_length'].first().rename('hh_size_reported_roster')

heads = (
    ind[ind['relationship_details'] == 1][['interview__key', 'sex']]
    .drop_duplicates('interview__key')
    .rename(columns={'sex': 'hh_head_sex'})
    .set_index('interview__key')
)

ind_agg = (
    ind.groupby('interview__key', as_index=False).agg(
        n_female        = ('b04',             lambda x: (x == 2).sum()),
        n_children      = ('age_dep',         lambda x: (x == 0).sum()),
        n_elderly       = ('age_dep',         lambda x: (x == 65).sum()),
        n_working_age   = ('age_dep',         lambda x: (x == 15).sum()),
        any_disability  = ('any_disability',  'max'),
        max_edu_isced   = ('ken_edu_isced11', 'max'),
        mean_age        = ('age_cur',         'mean'),
    )
)
ind_agg = ind_agg.merge(actual_hh_size,   on='interview__key', how='left')
ind_agg = ind_agg.merge(reported_hh_size, on='interview__key', how='left')
ind_agg = ind_agg.merge(heads,            on='interview__key', how='left')

ind_agg['hh_size_roster_mismatch'] = (ind_agg['hh_size'] != ind_agg['hh_size_reported_roster']).astype(int)
ind_agg['dependency_ratio'] = np.where(
    ind_agg['n_working_age'] > 0,
    (ind_agg['n_children'] + ind_agg['n_elderly']) / ind_agg['n_working_age'],
    np.nan,
)

print(f"Individual aggregates : {ind_agg.shape[0]:,} rows × {ind_agg.shape[1]} cols")
print(f"hh_size mismatches (actual vs reported roster): {ind_agg['hh_size_roster_mismatch'].sum()}")
print(f"hh_head_sex coverage: {ind_agg['hh_head_sex'].notna().mean()*100:.1f}%  "
      f"(was {ind['hhh_sex'].notna().mean()*100:.1f}% under the old 'hhh_sex' source)")

Individual aggregates : 21,347 rows × 13 cols
hh_size mismatches (actual vs reported roster): 4
hh_head_sex coverage: 99.7%  (was 26.3% under the old 'hhh_sex' source)


In [12]:
# ── 1.8  Dwelling → household aggregates (FIXED + EXPANDED) ──────────────
# Fix (kept): filter to spine keys before groupby to prevent row inflation.
# Fix: pull in d12 — the TRUE tenure variable (4-category OWNS/RENTS/
#      NO-RENT-CONSENT/SQUATTING; distribution 61.4/32.5/5.1/1.0% is highly
#      plausible nationally) — which the old pipeline never captured at all.
#      What used to be labelled 'hh_tenure_type' was actually d01, whose
#      distribution (85.5/11.9/2.2/0.3%... decaying) matches D00 "how many
#      dwelling units does this household occupy?", not tenure — see the
#      household-frame cell below for that rename.
# Fix: pull in d04 (attached/detached), also never captured.
# Fix: d11 and d11_2 were labelled 'dw_rooms'/'dw_bedrooms' but are BINARY
#      (2 values each) — cannot be room counts. Kept as honestly-named
#      'unverified' columns instead of a plausible-sounding wrong label.
#      d08_1 (0–15, 13 distinct values) is a much better room/bedroom-count
#      candidate but isn't confirmed against a codebook either — also kept
#      unverified. d13/d14/d15/d16 likewise kept, unverified.

dw = dfs['dwelling'].copy()
hh_keys = set(hh['interview__key'].dropna())

dw_spine = dw[dw['interview__key'].isin(hh_keys)]
print(f"Dwelling rows after spine filter : {len(dw_spine):,}  "
      f"(dropped {len(dw) - len(dw_spine)} stray keys)")

dw_agg = (
    dw_spine.sort_values('interview__key')
      .groupby('interview__key', as_index=False)
      .agg(
          dw_type                 = ('d03',  'first'),
          dw_attached             = ('d04',  'first'),
          dw_wall_mat             = ('d08',  'first'),
          dw_roof_mat             = ('d09',  'first'),
          dw_floor_mat            = ('d10',  'first'),
          dw_area_m2              = ('d11_1','first'),
          dw_approved             = ('d05',  'first'),
          dw_has_planning         = ('d06',  'first'),
          dw_in_hazard_zone       = ('d07',  'first'),
          dwelling_tenure_type    = ('d12',  'first'),   # NEW — real tenure var
          dw_raw_d11_unverified   = ('d11',  'first'),   # was wrongly 'dw_rooms'
          dw_raw_d11_2_unverified = ('d11_2','first'),   # was wrongly 'dw_bedrooms'
          dw_raw_d08_1_unverified = ('d08_1','first'),   # room/bedroom candidate
          dw_raw_d13_unverified   = ('d13',  'first'),
          dw_raw_d14_unverified   = ('d14',  'first'),
          dw_raw_d15_unverified   = ('d15',  'first'),
          dw_raw_d16_unverified   = ('d16',  'first'),
          dw_units_enumerated     = ('d03',  'count'),
      )
)

assert len(dw_agg) <= len(hh), \
    f"dw_agg still larger than spine: {len(dw_agg):,}"

# Labels + owner-occupier flag derived from the confirmed d12 variable
TENURE_LABELS = {1: 'Owns', 2: 'Rents/leases', 3: 'No rent (owner consent)', 4: 'No rent (squatting)'}
dw_agg['hh_tenure_type_label'] = dw_agg['dwelling_tenure_type'].map(TENURE_LABELS)
dw_agg['is_owner_occupier'] = (dw_agg['dwelling_tenure_type'] == 1).astype('Int64')
dw_agg.loc[dw_agg['dwelling_tenure_type'].isna(), 'is_owner_occupier'] = pd.NA

print(f"Dwelling aggregates   : {dw_agg.shape[0]:,} rows × {dw_agg.shape[1]} cols")
print(dw_agg['hh_tenure_type_label'].value_counts(normalize=True).round(3))

Dwelling rows after spine filter : 25,108  (dropped 8 stray keys)
Dwelling aggregates   : 21,346 rows × 21 cols
hh_tenure_type_label
Owns                      0.6150
Rents/leases              0.3250
No rent (owner consent)   0.0500
No rent (squatting)       0.0100
Name: proportion, dtype: float64


In [13]:
# ── 1.9  Land parcels → household aggregates ─────────────────────────────
# Same precaution applied: filter lp to spine keys first.
# (lp has 9,710 unique keys — 3 are outside the spine per join feasibility.)

lp = dfs['land_parcels'].copy()
lp_spine = lp[lp['interview__key'].isin(hh_keys)]
print(f"Land parcel rows after spine filter : {len(lp_spine):,}  "
      f"(dropped {len(lp) - len(lp_spine)} stray keys)")

lp_agg = (
    lp_spine.groupby('interview__key', as_index=False).agg(
        lp_n_parcels       = ('land_parcels__id', 'count'),
        lp_primary_tenure  = ('i01_3',  'first'),
        lp_has_title       = ('i05',    lambda x: int((x == 1).any())),
        lp_primary_use     = ('i06',    'first'),
        lp_any_dispute     = ('i08',    'max'),
        lp_any_registered  = ('i10',    'max'),
        lp_any_collateral  = ('i12',    'max'),
    )
)

print(f"Land parcel aggregates: {lp_agg.shape[0]:,} rows × {lp_agg.shape[1]} cols")


Land parcel rows after spine filter : 11,133  (dropped 3 stray keys)
Land parcel aggregates: 9,707 rows × 8 cols


In [14]:
# ── 1.9b  Fix nema_agg — deduplicate duplicate county code ────────────────
# nema_agg has 48 rows but only 47 counties.  Collapse on a01 by mean
# so the county join never produces a many-to-one expansion.

nema_agg = (
    nema_agg.groupby('a01', as_index=False).agg(
        nema_eia_applications = ('nema_eia_applications', 'sum'),
        nema_eia_approvals    = ('nema_eia_approvals',    'sum'),
        nema_processing_days  = ('nema_processing_days',  'mean'),
    )
)
assert len(nema_agg) <= 47, \
    f"nema_agg still has {len(nema_agg)} rows after dedup"
print(f"nema_agg after dedup  : {len(nema_agg)} rows")


nema_agg after dedup  : 47 rows


In [15]:
# ── 1.10  Build master analytical frame (FIXED) ───────────────────────────
master = hh.copy()

# HH-level joins
master = master.merge(ind_agg, on='interview__key', how='left', suffixes=('', '_i'))
master = master.merge(dw_agg,  on='interview__key', how='left', suffixes=('', '_d'))
master = master.merge(lp_agg,  on='interview__key', how='left', suffixes=('', '_l'))

# County-level joins
master['a01'] = master['a01'].astype('Int64')
for cdf in [county_agg, nema_agg, wsvc_agg, mort_agg, loan_agg, fin_agg,
            realest_agg, institutional_agg]:
    cdf['a01'] = cdf['a01'].astype('Int64')
    master = master.merge(cdf, on='a01', how='left')

# Corrected urban/rural direction (ground-truth checked in 1.3b: a07_1==2 is Urban)
master['is_urban'] = (master['a07_1'] == 2).astype('Int64')

# ── assertions ──────────────────────────────────────────────────────────
assert len(master) == len(hh), \
    f"Row inflation detected: {len(master):,} != {len(hh):,}"
assert master['interview__key'].nunique() == len(master), \
    "Duplicate interview__key values in master frame"

added_cols = [c for c in master.columns if c not in hh.columns]
print("=" * 60)
print("MASTER FRAME — BUILD COMPLETE (fixed)")
print("=" * 60)
print(f"  Rows    : {len(master):,}  (spine intact)")
print(f"  Columns : {master.shape[1]}  ({len(hh.columns)} base + {len(added_cols)} added)")
print(f"\n  master is ready.  Shape: {master.shape}")





layers = [
    ('household (base)',    hh.columns.tolist()),
    ('individual (agg)',    [c for c in ind_agg.columns  if c != 'interview__key']),
    ('dwelling (agg)',      [c for c in dw_agg.columns   if c != 'interview__key']),
    ('land_parcels (agg)',  [c for c in lp_agg.columns   if c != 'interview__key']),
    ('county_agg',          [c for c in county_agg.columns if c != 'a01']),
    ('nema_agg',            [c for c in nema_agg.columns   if c != 'a01']),
    ('wsvc_agg',            [c for c in wsvc_agg.columns   if c != 'a01']),
    ('mort_agg',            [c for c in mort_agg.columns   if c != 'a01']),
    ('loan_agg',            [c for c in loan_agg.columns   if c != 'a01']),
    ('fin_agg',             [c for c in fin_agg.columns    if c != 'a01']),
]
print(f"\n  {'Layer':<25} {'Cols':>6}")
print("  " + "─" * 33)
for name, cols in layers:
    print(f"  {name:<25} {len(cols):>6}")

high_miss = (
    master[added_cols].isna().mean()
    .pipe(lambda s: s[s > 0.50])
    .sort_values(ascending=False)
)
print()
if len(high_miss):
    print(f"  Added columns >50% missing ({len(high_miss)}):")
    for col, rate in high_miss.head(10).items():
        print(f"    {col:<42} {rate*100:.0f}%")
    if len(high_miss) > 10:
        print(f"    ... and {len(high_miss)-10} more")
else:
    print("  No added columns exceed 50% missing.")

print(f"\n  master is ready.  Shape: {master.shape}")

MASTER FRAME — BUILD COMPLETE (fixed)
  Rows    : 21,347  (spine intact)
  Columns : 457  (392 base + 65 added)

  master is ready.  Shape: (21347, 457)

  Layer                       Cols
  ─────────────────────────────────
  household (base)             392
  individual (agg)              12
  dwelling (agg)                20
  land_parcels (agg)             7
  county_agg                     5
  nema_agg                       3
  wsvc_agg                       5
  mort_agg                       4
  loan_agg                       3
  fin_agg                        3

  Added columns >50% missing (14):
    dw_raw_d13_unverified                      85%
    mort_interest_rate                         81%
    realest_n_establishments                   68%
    fin_avg_tenure_months                      64%
    mort_ltv_ratio                             64%
    inst_n_establishments                      63%
    mort_avg_term_years                        60%
    lp_any_dispute              

In [16]:
master.head()

,interview__key,interview__id,a01,countycode,a07_1,serial,a12,c01_1,c01_1other,c01_2,c01_2other,c01_3,c01_4,c01_5,c02_1,c02_1other,c02_2,c02_2other,c02_3,c02_4,...,nema_eia_approvals,nema_processing_days,wsvc_water_connections,wsvc_sewer_connections,wsvc_water_tariff,wsvc_service_quality,wsvc_n_providers,mort_interest_rate,mort_ltv_ratio,mort_avg_term_years,mort_n_providers,loan_avg_size,loan_outstanding,loan_n_providers,fin_portfolio,fin_avg_tenure_months,fin_n_financiers,realest_n_establishments,inst_n_establishments,is_urban
0,00-00-55-14,8a585d4dd71641b8a1348be6cc11e121,31,31,2,35312800,6,1,,2,,0,NaN,NaN,1,,2,,0,NaN,...,42.0000,30.0000,1592.0000,1231.0000,5000.0000,1.0000,2.0000,1.5333,18.6475,1.8000,3.0000,285184303.9000,647953187.0000,10.0000,398702274.0000,13.0000,2.0000,NaN,NaN,1
1,00-01-22-52,d155c88b64de40148cda8dd079c36baa,14,14,2,44136136,3,1,,1,,0,NaN,NaN,1,,1,,0,NaN,...,151.0000,5.0000,4491.0000,4166.0000,1000.0000,1.4000,5.0000,1.0000,100.0000,1.0000,4.0000,239184563.1246,1779292.7007,13.0000,13775733071.7700,15.0000,4.0000,NaN,NaN,1
2,00-02-11-67,b511ae4612704d9b9d395c901d975644,38,38,2,99694240,7,7,,5,,0,30.0000,1.0000,7,,5,,0,30.0000,...,123.0000,13.0000,0.0000,251.0000,0.0000,2.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,00-02-70-91,e1ff2d92960947d8adc30f55ff666b15,7,07,1,92351040,6,6,,5,,1,20.0000,1.0000,6,,5,,1,20.0000,...,5.0000,21.0000,300.0000,382.0000,3000.0000,1.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,00-02-78-46,522414c7575543ddb88782491aa5e190,14,14,2,26181404,4,1,,1,,0,NaN,NaN,1,,1,,0,NaN,...,151.0000,5.0000,4491.0000,4166.0000,1000.0000,1.4000,5.0000,1.0000,100.0000,1.0000,4.0000,239184563.1246,1779292.7007,13.0000,13775733071.7700,15.0000,4.0000,NaN,NaN,1


In [17]:
# ── 1.3b  Verify a07_1 coding BEFORE any rename or downstream use ─────────
# The VARIABLE_REGISTRY label above (1=Urban, 2=Rural) is an assumption,
# not yet confirmed against this raw extract. Check it against counties
# with unambiguous ground truth before it gets renamed to `urban_rural`
# and propagated into every downstream notebook.

# Nairobi (code 47) has no rural enumeration areas in the KHS frame —
# it must read as ~100% one value. Turkana (23) and Marsabit (10) are
# classic ASAL counties with real urban centres (Lodwar, Marsabit town)
# alongside large rural populations — they should show a genuine mix,
# not confirm direction on their own.
CHECK_COUNTIES = {47: 'Nairobi (must be ~100% urban)',
                   1:  'Mombasa (must be ~100% urban)',
                   23: 'Turkana (mixed — sanity check only)',
                   10: 'Marsabit (mixed — sanity check only)'}

print("a07_1 GROUND-TRUTH CHECK")
print("=" * 60)
for code, note in CHECK_COUNTIES.items():
    sub = hh.loc[hh['a01'] == code, 'a07_1']
    if sub.empty:
        print(f"  county {code:>2} — no rows found")
        continue
    dist = sub.value_counts(normalize=True).sort_index()
    dist_str = ', '.join(f"{int(v)}={p*100:.1f}%" for v, p in dist.items())
    print(f"  {code:>2} {note:<38} n={len(sub):<5} {dist_str}")

# Decisive check: whichever value dominates Nairobi is the TRUE "Urban" code.
nairobi_dist = hh.loc[hh['a01'] == 47, 'a07_1'].value_counts(normalize=True)
true_urban_code = int(nairobi_dist.idxmax())
registry_claim  = 1  # from VARIABLE_REGISTRY label above

print()
if true_urban_code == registry_claim:
    print(f"  ✓ CONFIRMED — a07_1=={registry_claim} is Urban, as the registry claims.")
else:
    print(f"  ⚠ REGISTRY LABEL IS WRONG.")
    print(f"    Registry claims Urban == {registry_claim}; Nairobi data shows "
          f"Urban == {true_urban_code} ({nairobi_dist.max()*100:.1f}% of Nairobi rows).")
    print(f"    Fix VARIABLE_REGISTRY['a07_1']['label'] and any downstream")
    print(f"    is_urban derivation to use a07_1 == {true_urban_code}, not {registry_claim}.")

a07_1 GROUND-TRUTH CHECK
  47 Nairobi (must be ~100% urban)          n=1059  2=100.0%
   1 Mombasa (must be ~100% urban)          n=425   2=100.0%
  23 Turkana (mixed — sanity check only)    n=368   1=56.0%, 2=44.0%
  10 Marsabit (mixed — sanity check only)   n=462   1=56.5%, 2=43.5%

  ⚠ REGISTRY LABEL IS WRONG.
    Registry claims Urban == 1; Nairobi data shows Urban == 2 (100.0% of Nairobi rows).
    Fix VARIABLE_REGISTRY['a07_1']['label'] and any downstream
    is_urban derivation to use a07_1 == 2, not 1.


In [18]:
#null count as percentage

master_null = master.isnull()
master_null_percent = master_null.mean()
master_null_percent


,0
interview__key,0.0000
interview__id,0.0000
a01,0.0000
countycode,0.0000
a07_1,0.0000
...,...
fin_avg_tenure_months,0.6404
fin_n_financiers,0.1861
realest_n_establishments,0.6752
inst_n_establishments,0.6347


In [19]:
# ── 1.3a  RENAME_MAP — raw column → clean column name ────────────────────
# Built once here; patched by MANUAL_FIXES below; consumed (not redefined)
# by the build/rename cell further down.

RENAME_MAP = {

    # ── Survey admin ──────────────────────────────────────────────────────────
    'interview__id'   : 'hh_uuid',
    'a12'             : 'interview_result',
    'tag'             : 'survey_tag',
    'rnd'             : 'sample_round',
    'selectedage'     : 'selected_respondent_age',

    # ── Identifiers & geography ───────────────────────────────────────────────
    'interview__key'  : 'hh_id',
    'a01'             : 'county_code',
    'countycode'      : 'county_code_str',
    'a07_1'           : 'urban_rural',
    'serial'          : 'hh_serial',
    'hhweight'        : 'hh_weight',

    # ── Water & sanitation (core) ─────────────────────────────────────────────
    'c01_1'           : 'water_src_main',
    'c01_2'           : 'water_collect_method',
    'c01_3'           : 'water_treated',
    'c01_4'           : 'water_dist_mins',
    'c02_1'           : 'water_src_secondary',
    'c04'             : 'toilet_type',
    'c05'             : 'has_handwash_facility',
    'c07'             : 'handwash_materials',
    'c14_1'           : 'spend_water_kes',
    'c01_1other'      : 'water_src_main_other',
    'c01_2other'      : 'water_collect_main_other',
    'c01_5'           : 'water_src_main_season',
    'c02_2'           : 'water_collect_secondary',
    'c02_2other'      : 'water_collect_secondary_other',
    'c02_3'           : 'water_secondary_treated',
    'c02_4'           : 'water_secondary_dist_mins',
    'c02_5'           : 'water_secondary_season',
    'c02_1other'      : 'water_src_secondary_other',
    'c03'             : 'n_toilet_facilities',
    'c03_other'       : 'toilet_facility_other',
    'c04other'        : 'toilet_type_other',
    'c06'             : 'n_hh_sharing_toilet',
    'c07_1'           : 'handwash_materials_other',
    'c08'             : 'has_bathhouse',

    # ── Energy ───────────────────────────────────────────────────────────────
    'c10'             : 'lighting_src',
    'c10_2'           : 'electricity_hrs_day',
    'c10_4'           : 'electricity_conn_type',
    'c11'             : 'cooking_fuel',
    'c12'             : 'cooking_stove_type',
    'c14_2'           : 'spend_electricity_kes',
    'c14_3'           : 'spend_energy_other_kes',
    'c10_1'           : 'lighting_src_other',
    'c10_3'           : 'electricity_supply_type',
    'c11_other'       : 'cooking_fuel_other',
    'c11_1'           : 'cooking_fuel_n_types',
    'c11_2'           : 'cooking_fuel_secondary',
    'c11_2_1'         : 'cooking_fuel_secondary_spend_kes',
    'c11_3'           : 'cooking_fuel_tertiary',
    'c11_4__1'        : 'cooking_fuel_wood_source_own',
    'c11_4__2'        : 'cooking_fuel_wood_source_buy',
    'c11_4__3'        : 'cooking_fuel_wood_source_collect',
    'c11_4__4'        : 'cooking_fuel_wood_source_gift',
    'c11_4__5'        : 'cooking_fuel_wood_source_other1',
    'c11_4__6'        : 'cooking_fuel_wood_source_other2',
    'c11_4__7'        : 'cooking_fuel_wood_source_other3',
    'c11_4__98'       : 'cooking_fuel_wood_source_dk',
    'c12_other'       : 'cooking_stove_other',
    'c12_1'           : 'n_cooking_spaces',
    'c12_1_1'         : 'cooking_space_dist_mins',
    'c12_2'           : 'cooking_space_indoor',
    'c12_3'           : 'cooking_stove_secondary',
    'c12_3_1'         : 'cooking_space_secondary_dist_mins',
    'c12_4'           : 'cooking_space_secondary_indoor',
    'c09__1'          : 'waste_coll_county',
    'c09__2'          : 'waste_coll_private',
    'c09__3'          : 'waste_coll_ngo',
    'c09__4'          : 'waste_coll_community',
    'c09__5'          : 'waste_coll_self',
    'c09__6'          : 'waste_coll_neighbour',
    'c09__7'          : 'waste_coll_other',
    'c09__96'         : 'waste_coll_other_specify',
    'c09_other'       : 'waste_coll_other_text',

    # ── Assets ───────────────────────────────────────────────────────────────
    'c13__1'          : 'owns_radio',
    'c13__2'          : 'owns_mobile',
    'c13__3'          : 'owns_tv',
    'c13__4'          : 'owns_computer',
    'c13__5'          : 'owns_motorcycle',
    'c13__6'          : 'owns_vehicle',
    'c13__7'          : 'owns_fridge',
    'c13__96'         : 'owns_other_asset',
    'c13_other'       : 'owns_other_asset_text',
    'internet'        : 'has_internet',

    # ── Dwelling (household module) ───────────────────────────────────────────
    'd01'             : 'hh_tenure_type',
    'd17'             : 'dwelling_ownership_doc',
    'd18__1'          : 'ownership_doc_title_deed',
    'd18__2'          : 'ownership_doc_allotment',
    'd18__3'          : 'ownership_doc_agreement',
    'd18__4'          : 'ownership_doc_rent_receipt',
    'd18__5'          : 'ownership_doc_will',
    'd18__6'          : 'ownership_doc_letter',
    'd18__7'          : 'ownership_doc_other',
    'd18__8'          : 'ownership_doc_none',
    'd19'             : 'n_hh_in_building',
    'd20__1'          : 'shared_facility_water',
    'd20__2'          : 'shared_facility_toilet',
    'd20__3'          : 'shared_facility_bathroom',
    'd20__4'          : 'shared_facility_kitchen',
    'd20__5'          : 'shared_facility_entrance',
    'd20__6'          : 'shared_facility_yard',
    'd20__7'          : 'shared_facility_parking',
    'd20__8'          : 'shared_facility_none',
    'd20__9'          : 'shared_facility_other',
    'd20__10'         : 'shared_facility_other2',
    'duration'        : 'tenancy_duration_cat',
    'year_occ'        : 'year_occupied_cat',
    'bf'              : 'building_floor_cat',

    # ── Income & expenditure / Section G / Section H (first pass — corrected below) ──
    'g01a'            : 'spend_food_kes',
    'g01b'            : 'spend_clothing_kes',
    'g01c'            : 'spend_education_kes',
    'g01d'            : 'spend_health_kes',
    'g01e'            : 'spend_transport_kes',
    'g01f'            : 'spend_comms_kes',
    'g01g'            : 'spend_recreation_kes',
    'g01h'            : 'spend_housing_kes',
    'g01i'            : 'spend_energy_kes',
    'g01j'            : 'spend_other_kes',
    'g01k'            : 'spend_remittances_kes',
    'g02'             : 'pays_rent',
    'g02_1'           : 'rent_monthly_kes',
    'g03'             : 'tenure_type',
    'g04'             : 'owns_other_property',
    'g05__1'          : 'prob_overcrowding',
    'g05__2'          : 'prob_poor_water',
    'g05__3'          : 'prob_poor_sanitation',
    'g05__4'          : 'prob_poor_drainage',
    'g05__5'          : 'prob_poor_road',
    'g05__6'          : 'prob_insecurity',
    'g05__7'          : 'prob_high_cost',
    'g05__8'          : 'prob_poor_structure',
    'g05__9'          : 'prob_crime',
    'g05__10'         : 'prob_other',
    'g06__1'          : 'aspire_buy_land',
    'g06__2'          : 'aspire_build',
    'g06__3'          : 'aspire_buy_house',
    'g06__4'          : 'aspire_rent_better',
    'g06__5'          : 'aspire_renovate',
    'g06__6'          : 'aspire_move_county',
    'g06__7'          : 'aspire_stay_improve',
    'g06__8'          : 'aspire_no_change',
    'g06__9'          : 'aspire_other',
    'g06__10'         : 'aspire_other2',
    'h01'             : 'perc_structure',
    'h02'             : 'perc_roof',
    'h03'             : 'perc_walls',
    'h04'             : 'perc_floor',
    'h05'             : 'perc_ventilation',
    'h06'             : 'perc_lighting',
    'h07'             : 'perc_water',
    'h08'             : 'perc_sanitation',
    'h09'             : 'perc_waste',
    'h10'             : 'perc_security',
    'h11'             : 'perc_overall',

    # ── Land ownership ────────────────────────────────────────────────────────
    'i00'             : 'owns_land',

    # ── Tenure & mobility ──────────────────────────────────────────────────────
    'j04_1'           : 'is_owner_occupier',
    'j05'             : 'has_title_doc',
    'j09'             : 'housing_cost_burden',
    'j10'             : 'missed_payment',
    'j11'             : 'eviction_risk',
    'j12_1'           : 'yrs_in_dwelling',
    'j13'             : 'satisfied_tenure',
    'j02'             : 'ever_owned_dwelling',
    'j03'             : 'prev_tenure_type',
    'j04_2'           : 'tenure_is_informal',
    'j06_1'           : 'moved_last_5yrs',
    'j06_2__1'        : 'move_reason_work',
    'j06_2__2'        : 'move_reason_family',
    'j06_2__3'        : 'move_reason_eviction',
    'j06_2__4'        : 'move_reason_disaster',
    'j06_2__5'        : 'move_reason_cost',
    'j06_2__6'        : 'move_reason_better_house',
    'j06_2__7'        : 'move_reason_other',
    'j06_2__96'       : 'move_reason_other_specify',
    'j06_2_other'     : 'move_reason_other_text',
    'j07'             : 'was_displaced',
    'j08'             : 'displacement_cause',
    'j12_2'           : 'prev_dwelling_tenure',
    'j12_2other'      : 'prev_dwelling_tenure_other',
    'j14'             : 'wants_to_own',
    'j15'             : 'applied_for_housing',
    'j16__1'          : 'housing_program_nhc',
    'j16__2'          : 'housing_program_county',
    'j16__3'          : 'housing_program_employer',
    'j16__4'          : 'housing_program_sacco',
    'j16__5'          : 'housing_program_ngo',
    'j16__6'          : 'housing_program_other',
    'j16__96'         : 'housing_program_other_specify',
    'j16_1'           : 'housing_program_other_text',
    'j17'             : 'willing_to_relocate',
    'j18'             : 'relocation_preference',
    'j19__1'          : 'barrier_cost',
    'j19__2'          : 'barrier_no_land',
    'j19__3'          : 'barrier_no_loan',
    'j19__4'          : 'barrier_bureaucracy',
    'j19__5'          : 'barrier_tenure_insecurity',
    'j19__6'          : 'barrier_other',
    'j19__96'         : 'barrier_other_specify',
    'j19_1'           : 'barrier_other_text',
    'j20'             : 'aware_affordable_housing',
    'j21'             : 'applied_affordable_housing',
    'j22__1'          : 'loan_barrier_income',
    'j22__2'          : 'loan_barrier_collateral',
    'j22__3'          : 'loan_barrier_high_rate',
    'j22__4'          : 'loan_barrier_short_term',
    'j22__5'          : 'loan_barrier_documentation',
    'j22__96'         : 'loan_barrier_other_specify',
    'j22_1'           : 'loan_barrier_other_text',

    # ── Rental module ───────────────────────────────────────────────────────────
    'k02'             : 'has_written_lease',
    'k05'             : 'rent_actual_kes',
    'k09'             : 'lease_type',
    'k21'             : 'rent_arrears',
    'min_rent'        : 'psu_min_rent_kes',
    'k01'             : 'rental_market_type',
    'k03'             : 'rent_negotiated',
    'k04'             : 'landlord_type',
    'k04_1'           : 'landlord_type_other',
    'k06_1'           : 'rent_includes_water',
    'k06_2'           : 'water_charge_in_rent_kes',
    'k07'             : 'rent_includes_electricity',
    'k08_1'           : 'rent_includes_security',
    'k08_2'           : 'security_charge_type',
    'k08_3'           : 'security_charge_kes',
    'k10'             : 'rent_deposit_kes',
    'k11'             : 'rent_payment_method',
    'k11_1'           : 'rent_payment_method_other',
    'k12'             : 'rent_collection_method',
    'k12_1'           : 'rent_collection_other',
    'k13'             : 'receipt_given',
    'k14'             : 'rent_increase_last_yr_kes',
    'k15'             : 'has_rent_dispute',
    'k16'             : 'desired_dwelling_type',
    'k16_1'           : 'desired_dwelling_type_other',
    'k17'             : 'desired_tenure',
    'k18'             : 'desired_rent_kes',
    'k19'             : 'desired_location_county',
    'k19_1'           : 'desired_location_area',
    'k20'             : 'plans_to_buy',
    'k22'             : 'buy_readiness',
    'k22_1'           : 'buy_readiness_other',
    'k23'             : 'buy_finance_source',
    'k24'             : 'preferred_buy_county',
    'k24_1'           : 'preferred_buy_area',
    'k25'             : 'willingness_to_pay_kes',
    'k26__1'          : 'savings_for_housing',
    'k26__2'          : 'sacco_for_housing',
    'k26__3'          : 'bank_loan_for_housing',
    'k26__4'          : 'family_help_for_housing',
    'k26__5'          : 'employer_scheme_for_housing',
    'k26__6'          : 'govt_scheme_for_housing',
    'k26__7'          : 'ngo_for_housing',
    'k26__8'          : 'no_savings_for_housing',
    'k26__96'         : 'other_savings_for_housing',
    'k26_other'       : 'other_savings_text',
    'k27'             : 'savings_amount_kes',
    'k28'             : 'savings_duration_months',
    'k29'             : 'aware_nhf',
    'k30'             : 'nhf_info_source',
    'k30_1'           : 'nhf_info_source_other',
    'k31'             : 'contributed_nhf',
    'k32'             : 'nhf_contribution_months',
    'k32_1'           : 'nhf_noncontrib_reason',
    'k33'             : 'utility_responsibility',
    'k33_other'       : 'utility_responsibility_other',
    'k34'             : 'landlord_maintains',
    'k35'             : 'tenant_can_renovate',
    'k36'             : 'eviction_notice_months',
    'k36_1'           : 'eviction_reason_other',
    'k37'             : 'has_sublease',
    'k38'             : 'sublease_type',
    'k39'             : 'end_tenancy_reason',
    'k39_other'       : 'end_tenancy_reason_other',

    # ── Owned dwelling module ───────────────────────────────────────────────────
    'l07'             : 'dwelling_yr_built',
    'l13'             : 'mortgage_repayment_kes',
    'l14'             : 'dwelling_value_kes',
    'l15'             : 'imputed_rent_kes',
    'l19'             : 'plot_size_decimals',
    'l01_1'           : 'acquisition_method',
    'l01_1a'          : 'acquisition_method_other',
    'l01_2'           : 'construction_method',
    'l01_2a'          : 'construction_method_other',
    'l02__1'          : 'finance_own_savings',
    'l02__2'          : 'finance_bank_loan',
    'l02__3'          : 'finance_sacco',
    'l02__4'          : 'finance_employer',
    'l02__5'          : 'finance_family',
    'l02__6'          : 'finance_govt',
    'l02__7'          : 'finance_ngo',
    'l02__8'          : 'finance_other',
    'l02__96'         : 'finance_other_specify',
    'l08'             : 'dwelling_yr_surveyed',
    'l09'             : 'dwelling_completion_pct',
    'l10'             : 'dwelling_yr_renovated_major',
    'l11'             : 'has_outstanding_mortgage',
    'l12'             : 'mortgage_yr_taken',
    'l16__1'          : 'challenge_cost',
    'l16__2'          : 'challenge_land',
    'l16__3'          : 'challenge_approvals',
    'l16__4'          : 'challenge_materials',
    'l16__5'          : 'challenge_labour',
    'l16__6'          : 'challenge_finance',
    'l16__7'          : 'challenge_design',
    'l16__8'          : 'challenge_security',
    'l16__9'          : 'challenge_dispute',
    'l16__10'         : 'challenge_infrastructure',
    'l16__11'         : 'challenge_other1',
    'l16__12'         : 'challenge_other2',
    'l16__13'         : 'challenge_other3',
    'l16__96'         : 'challenge_other_specify',
    'l16_a'           : 'challenge_other_text',
    'l16b__1'         : 'satisfied_structure',
    'l16b__2'         : 'satisfied_size',
    'l16b__3'         : 'satisfied_location',
    'l16b__4'         : 'satisfied_cost',
    'l16b__5'         : 'satisfied_security',
    'l16b__6'         : 'satisfied_utilities',
    'l16b__7'         : 'satisfied_design',
    'l16b__8'         : 'satisfied_neighbourhood',
    'l16b__9'         : 'satisfied_tenure_ownership',
    'l16b__10'        : 'satisfied_overall',
    'l16b__11'        : 'satisfied_other1',
    'l16b__12'        : 'satisfied_other2',
    'l16b__13'        : 'satisfied_other3',
    'l19a'            : 'plot_size_unit_other',
    'l20_a'           : 'plot_boundary_north_m',
    'l20_b'           : 'plot_boundary_east_m',
    'l20_c'           : 'plot_boundary_other_m',
    'l21'             : 'had_renovation',
    'l22_1__1'        : 'renovation_roofing',
    'l22_1__2'        : 'renovation_walls',
    'l22_1__3'        : 'renovation_flooring',
    'l22_1__4'        : 'renovation_plumbing',
    'l22_1__5'        : 'renovation_electrical',
    'l22_1__6'        : 'renovation_extension',
    'l22_1__96'       : 'renovation_other_specify',
    'l22_1a'          : 'renovation_other_text',
    'l23'             : 'renovation_self_financed',
    'l24'             : 'renovation_cost_kes',
    'l25'             : 'plans_further_renovation',
    'l26'             : 'plans_sell',
    'l27__1'          : 'loss_reason_fire',
    'l27__2'          : 'loss_reason_flood',
    'l27__3'          : 'loss_reason_demolition',
    'l27__4'          : 'loss_reason_eviction',
    'l27__5'          : 'loss_reason_dispute',
    'l27__6'          : 'loss_reason_other',
    'l27__96'         : 'loss_reason_other_specify',
    'l27_a'           : 'loss_reason_other_text',
    'l28'             : 'dwelling_yr_last_renovated',
    'l29'             : 'has_insurance',
    'l30'             : 'insurance_type',
    'l31'             : 'insurance_provider',
    'l32'             : 'insurance_premium_kes',

    # ── Environment & hazards ────────────────────────────────────────────────────
    'e01'             : 'waste_disposal_method',
    'e05'             : 'near_waste_dump',
    'e06'             : 'flood_exposure',
    'e07'             : 'erosion_exposure',
    'e08'             : 'terrain_type',
    'e02__1'          : 'near_factory',
    'e02__2'          : 'near_quarry',
    'e02__3'          : 'near_powerline',
    'e02__4'          : 'near_pipeline',
    'e02__5'          : 'near_railway',
    'e02__6'          : 'near_road',
    'e02__7'          : 'near_floodplain',
    'e02__8'          : 'near_landfill',
    'e02__9'          : 'near_other_hazard',
    'e02__96'         : 'near_hazard_other_specify',
    'e02_1'           : 'near_hazard_other_text',
    'e03'             : 'waste_collection_freq',
    'e03_1'           : 'waste_collection_other',
    'e04'             : 'waste_collection_days_per_wk',
    'e04_1'           : 'waste_collection_schedule_other',
    'e09__1'          : 'sanitation_flush_sewer',
    'e09__2'          : 'sanitation_flush_septic',
    'e09__3'          : 'sanitation_flush_pit',
    'e09__4'          : 'sanitation_vip_latrine',
    'e09__5'          : 'sanitation_pit_slab',
    'e09__6'          : 'sanitation_pit_no_slab',
    'e09__7'          : 'sanitation_composting',
    'e09__8'          : 'sanitation_hanging',
    'e09__9'          : 'sanitation_improved',
    'e09__10'         : 'sanitation_bucket',
    'e09__11'         : 'sanitation_open',
    'e09__12'         : 'sanitation_other',
    'e09__13'         : 'sanitation_none',

    # ── Derived / computed (household) ────────────────────────────────────────
    'prop_util'       : 'util_income_ratio',
    'med_prop'        : 'cty_med_util_ratio',
    'utilities'       : 'pays_utilities',
    'ctymin_ut'       : 'cty_min_utility_kes',
    'med_brms'        : 'cty_med_bedrooms',
    'sf'              : 'is_slum',
    'pln'             : 'settlement_plan_status',

    # ── Dwelling aggregates ───────────────────────────────────────────────────
    'dw_type'               : 'dw_type',
    'dw_wall_material'      : 'dw_wall_mat',
    'dw_roof_material'      : 'dw_roof_mat',
    'dw_floor_material'     : 'dw_floor_mat',
    'dw_n_rooms'            : 'dw_rooms',
    'dw_floor_area_m2'      : 'dw_area_m2',
    'dw_n_bedrooms'         : 'dw_bedrooms',
    'dw_approved'           : 'dw_approved',
    'dw_planning_ok'        : 'dw_has_planning',
    'dw_hazard_zone'        : 'dw_in_hazard_zone',
    'dw_n_units_enumerated' : 'dw_units_count',

    # ── Individual aggregates ─────────────────────────────────────────────────
    'hh_size'         : 'hh_size',
    'hhh_sex'         : 'hh_head_sex',
    'any_disability'  : 'has_disability',
    'max_edu_isced'   : 'max_edu_isced',
    'mean_age'        : 'mean_age',
    'dependency_ratio': 'dependency_ratio',
    'n_female'        : 'n_female',
    'n_children'      : 'n_children',
    'n_elderly'       : 'n_elderly',
    'n_working_age'   : 'n_working_age',

    # ── Land parcel aggregates ────────────────────────────────────────────────
    'lp_n_parcels'      : 'lp_n_parcels',
    'lp_primary_tenure' : 'lp_tenure_type',
    'lp_has_title'      : 'lp_has_title',
    'lp_primary_use'    : 'lp_land_use',
    'lp_any_dispute'    : 'lp_has_dispute',
    'lp_any_registered' : 'lp_is_registered',
    'lp_any_collateral' : 'lp_used_as_collateral',

    # ── County / NEMA / Water / Mortgage / Loan / Financier aggregates ────────
    'cty_housing_stock'           : 'cty_housing_stock',
    'cty_housing_backlog'         : 'cty_housing_backlog',
    'cty_planning_staff'          : 'cty_planning_staff',
    'cty_has_housing_policy'      : 'cty_has_housing_policy',
    'cty_building_approval_system': 'cty_approval_system',
    'nema_eia_applications' : 'nema_eia_apps',
    'nema_eia_approvals'    : 'nema_eia_approvals',
    'nema_processing_days'  : 'nema_proc_days',
    'wsvc_water_connections' : 'wsvc_water_conns',
    'wsvc_sewer_connections' : 'wsvc_sewer_conns',
    'wsvc_water_tariff'      : 'wsvc_tariff',
    'wsvc_service_quality'   : 'wsvc_quality',
    'wsvc_n_providers'       : 'wsvc_n_providers',
    'mort_interest_rate'  : 'mort_rate',
    'mort_ltv_ratio'      : 'mort_ltv',
    'mort_avg_term_years' : 'mort_term_yrs',
    'mort_n_providers'    : 'mort_n_providers',
    'loan_avg_size'    : 'loan_avg_kes',
    'loan_outstanding' : 'loan_outstanding_kes',
    'loan_n_providers' : 'loan_n_providers',
    'fin_portfolio'         : 'fin_portfolio_kes',
    'fin_avg_tenure_months' : 'fin_tenure_months',
    'fin_n_financiers'      : 'fin_n_financiers',

    # ── Section G distances (CORRECTED — overrides the wrong spend_*_kes above) ──
    'g01a' : 'dist_primary_school_m',
    'g01b' : 'dist_secondary_school_m',
    'g01c' : 'dist_police_station_m',
    'g01d' : 'dist_health_facility_m',
    'g01e' : 'dist_bus_stop_m',
    'g01f' : 'dist_shopping_centre_m',
    'g01g' : 'dist_worship_centre_m',
    'g01h' : 'dist_recreation_park_m',
    'g01i' : 'dist_market_m',
    'g01j' : 'dist_social_hall_m',
    'g01k' : 'dist_pickup_point_m',

    # ── Section H disability/accessibility (CORRECTED — overrides perc_* above, h01-h09) ──
    'h01' : 'access_mobility_rating',
    'h02' : 'access_safety_rating',
    'h03' : 'access_lighting_satisfaction',
    'h04' : 'access_wide_doorways',
    'h05' : 'access_grab_bars',
    'h06' : 'access_switch_height',
    'h07' : 'access_entrance_ramp',
    'h08' : 'access_lever_handles',
    'h09' : 'access_elevator',

    # ── Section G road/utilities (CORRECTED — overrides pays_rent/tenure_type/etc above) ──
    'g02'   : 'road_all_weather_flag',
    'g02_1' : 'road_dist_m',
    'g03'   : 'road_surface_type',
    'g04'   : 'road_has_street_lights',
    'g05__1'  : 'govt_provides_street_lighting',
    'g05__2'  : 'govt_provides_walkways',
    'g05__3'  : 'govt_provides_cycling_paths',
    'g05__4'  : 'govt_provides_drainage',
    'g05__5'  : 'govt_provides_sewerage',
    'g05__6'  : 'govt_provides_water',
    'g05__7'  : 'govt_provides_roads',
    'g05__8'  : 'govt_provides_health_facilities',
    'g05__9'  : 'govt_provides_garbage_collection',
    'g05__10' : 'govt_provides_recreation',
    'g06__1'  : 'govt_functional_street_lighting',
    'g06__2'  : 'govt_functional_walkways',
    'g06__3'  : 'govt_functional_cycling_paths',
    'g06__4'  : 'govt_functional_drainage',
    'g06__5'  : 'govt_functional_sewerage',
    'g06__6'  : 'govt_functional_water',
    'g06__7'  : 'govt_functional_roads',
    'g06__8'  : 'govt_functional_health_facilities',
    'g06__9'  : 'govt_functional_garbage_collection',
    'g06__10' : 'govt_functional_recreation',
}

print(f"RENAME_MAP built: {len(RENAME_MAP)} unique entries "
      f"(42 keys appear twice above — g/h duplicate-key self-correction, later value wins)")

RENAME_MAP built: 443 unique entries (42 keys appear twice above — g/h duplicate-key self-correction, later value wins)


In [20]:
# ── 1.3c  Extend the a07_1-style ground-truth check to the rest of RENAME_MAP ──
# Cell 18 proved a07_1's direction was backwards by checking it against
# known ground truth (Nairobi/Mombasa). The same kind of check — does the
# empirical cardinality/range match what the label claims? — turns up
# several more mislabeled columns that RENAME_MAP's duplicate-key issue
# (see below) never touches, because they were never duplicated in the
# first place; they were just wrong once and stayed wrong.

MANUAL_FIXES = {
    # col   : (new_name, reason)
    'h10'  : ('access_smoke_alarms',           'was perc_security — Section H is DISABILITY, not perception; 3-cat Yes/No/N-A matches smoke-alarm question'),
    'h11'  : ('access_kitchen_accommodations', 'was perc_overall — same Section H issue; 3-cat matches kitchen-accommodation question'),
    'd01'  : ('n_dwelling_units_occupied',     'was hh_tenure_type — dist. 85.5/11.9/2.2/0.3%...decaying matches D00 "how many dwelling units", not tenure (4-cat)'),
    'k03'  : ('tenancy_duration_yrs_k03',      'was rent_negotiated — range 0-58, 42 distinct values; matches K03 "how long have you stayed here" (years)'),
    'e05'  : ('dist_nearest_hazard_m_unverified', 'was near_waste_dump (binary) — range 0-15,000, 68 distinct values; a distance, not a flag. Target hazard unconfirmed.'),
    'sf'   : ('settlement_type_code_unverified',  'was is_slum (binary) — has 8 distinct values (1-9), not binary. True categories unconfirmed.'),
    'j04_1': ('j04_1_raw_unverified',          'was is_owner_occupier — 3 categories, does not align with real tenure var d12. Likely J04_1 "comfortable residing here?" but unconfirmed.'),
}

print("MANUAL FIXES (beyond the RENAME_MAP duplicate-key self-correction)")
print("=" * 78)
for raw, (new_name, reason) in MANUAL_FIXES.items():
    if raw in hh.columns:
        s = hh[raw]
        print(f"  {raw:8s} n_unique={s.nunique():<4} missing%={s.isna().mean()*100:5.1f}  -> {new_name}")
        print(f"           {reason}")
    print()

# Apply on top of RENAME_MAP so the build cell below picks them up
for raw, (new_name, _) in MANUAL_FIXES.items():
    RENAME_MAP[raw] = new_name

is_urban_note = (
    "is_urban will be derived as (a07_1 == 2) per the confirmed ground-truth "
    "check above, NOT (a07_1 == 1) as the original registry assumed."
)
print(is_urban_note)

MANUAL FIXES (beyond the RENAME_MAP duplicate-key self-correction)
  h10      n_unique=3    missing%=  0.0  -> access_smoke_alarms
           was perc_security — Section H is DISABILITY, not perception; 3-cat Yes/No/N-A matches smoke-alarm question

  h11      n_unique=3    missing%=  0.0  -> access_kitchen_accommodations
           was perc_overall — same Section H issue; 3-cat matches kitchen-accommodation question

  d01      n_unique=7    missing%=  0.0  -> n_dwelling_units_occupied
           was hh_tenure_type — dist. 85.5/11.9/2.2/0.3%...decaying matches D00 "how many dwelling units", not tenure (4-cat)

  k03      n_unique=42   missing%= 67.5  -> tenancy_duration_yrs_k03
           was rent_negotiated — range 0-58, 42 distinct values; matches K03 "how long have you stayed here" (years)

  e05      n_unique=68   missing%=  0.0  -> dist_nearest_hazard_m_unverified
           was near_waste_dump (binary) — range 0-15,000, 68 distinct values; a distance, not a flag. Target hazard u

In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL: Rename master → clean column names (build master_clean)
# ══════════════════════════════════════════════════════════════════════════════
#
# RENAME_MAP is NOT redefined here — it is the same dict object built in
# cell 1.3 (Variable Registry) and patched two cells above (MANUAL_FIXES),
# carrying forward every empirically-confirmed fix: the g/h duplicate-key
# self-fix, plus h10, h11, d01, k03, e05, sf, and j04_1. A previous version
# of this cell redeclared RENAME_MAP from scratch, which silently discarded
# the MANUAL_FIXES patch and reverted those seven columns to their original
# wrong labels — that duplicate cell has been removed. Saving to parquet
# happens in a separate cell below, AFTER the pre-save verification passes.

assert 'RENAME_MAP' in dir(), \
    "RENAME_MAP not found — run the Variable Registry and MANUAL_FIXES cells before this one."
assert RENAME_MAP.get('d01') == 'n_dwelling_units_occupied', \
    ("RENAME_MAP looks unpatched (d01 still maps to something other than "
     "n_dwelling_units_occupied) — did the MANUAL_FIXES cell run before this one?")

# ── 1. Pre-flight: detect collisions before touching the frame ────────────────
raw_to_drop = [
    raw for raw, clean in RENAME_MAP.items()
    if raw in master.columns and clean in master.columns and raw != clean
]
rename_safe = {
    raw: clean for raw, clean in RENAME_MAP.items()
    if raw in master.columns and raw not in raw_to_drop
}

if raw_to_drop:
    print(f"⚠  Collision — dropping {len(raw_to_drop)} raw col(s) "
          f"whose clean name already exists in master:")
    for r in raw_to_drop:
        print(f"   drop '{r}'  →  keep existing '{RENAME_MAP[r]}'")
else:
    print("✓  No collisions detected.")

# ── 2. Apply rename ───────────────────────────────────────────────────────────
master_clean = (
    master
    .drop(columns=raw_to_drop)
    .rename(columns=rename_safe)
)

assert master_clean.columns.is_unique, \
    f"Duplicate columns remain: {[c for c,n in Counter(master_clean.columns).items() if n>1]}"

print(f"\nColumns renamed  : {len(rename_safe)}")
print(f"Collisions dropped: {len(raw_to_drop)}")
print(f"Final shape      : {master_clean.shape}")
unmapped = [c for c in master_clean.columns if c not in rename_safe.values()]
print(f"Pass-through cols: {len(unmapped)}  (already clean or not in map)")


✓  No collisions detected.

Columns renamed  : 433
Collisions dropped: 0
Final shape      : (21347, 457)
Pass-through cols: 24  (already clean or not in map)


In [22]:


# ── 2b.  Sentinel / placeholder-code cleanup (NEW) ──────────────────────────
# Found by inspecting the actual value distributions in master_clean, not
# assumed from the codebook. Each of these is a "not applicable / don't
# know" placeholder that was left in as a literal number instead of being
# converted to NaN — left as-is, they silently distort any mean/regression
# that touches the column.
#
#   dwelling_yr_last_renovated : -1 for 9,507 rows (44.5%) — "never
#                                 renovated" sentinel, not a real year.
#   mortgage_yr_taken          : values of 1 and 98 — don't-know/NA codes,
#                                 not calendar years.
#   dwelling_yr_built          : one row = 0 — invalid year.
#   security_charge_kes        : one row negative — data-entry error.
#
# water_dist_mins is NOT auto-corrected here (max=1000 min could be a
# genuine remote-area value) — flagged for manual review only.

SENTINEL_FIXES = {
    'dwelling_yr_last_renovated': lambda s: s.mask(s == -1),
    'mortgage_yr_taken':          lambda s: s.mask(s < 1900),
    'dwelling_yr_built':          lambda s: s.mask((s < 1900) | (s == 0)),
    'security_charge_kes':        lambda s: s.mask(s < 0),
}

print("Sentinel cleanup:")
for col, fix in SENTINEL_FIXES.items():
    if col not in master_clean.columns:
        print(f"  {col:<28} NOT FOUND — skipped")
        continue
    before_bad = master_clean[col].isna().sum()
    master_clean[col] = fix(master_clean[col])
    after_bad = master_clean[col].isna().sum()
    print(f"  {col:<28} nulled {after_bad - before_bad:>5} sentinel value(s)  "
          f"(null% now {master_clean[col].isna().mean()*100:.1f})")

# water_dist_mins — flag only, no auto-fix (plausibly real in remote areas)
if 'water_dist_mins' in master_clean.columns:
    n_extreme = (master_clean['water_dist_mins'] > 180).sum()
    print(f"\n⚠  water_dist_mins: {n_extreme} households report >180 minutes "
          f"(3+ hrs) one-way — not auto-corrected, review before using as a "
          f"continuous predictor without capping/winsorizing.")


Sentinel cleanup:
  dwelling_yr_last_renovated   nulled  9507 sentinel value(s)  (null% now 82.9)
  mortgage_yr_taken            nulled     6 sentinel value(s)  (null% now 98.9)
  dwelling_yr_built            nulled     1 sentinel value(s)  (null% now 68.1)
  security_charge_kes          nulled     1 sentinel value(s)  (null% now 96.7)

⚠  water_dist_mins: 71 households report >180 minutes (3+ hrs) one-way — not auto-corrected, review before using as a continuous predictor without capping/winsorizing.


In [23]:
# ── Pre-save verification: confirm the known bugs are actually gone ──────
# Runs AFTER master_clean exists (previous cell) and BEFORE the save
# (next cell) — this is the checkpoint that stops a bad file from ever
# reaching disk. Do not comment this out; if it fails, fix the cause,
# don't skip the check.

BUG_CHECK = ['spend_food_kes', 'perc_structure', 'perc_security', 'perc_overall',
             'prob_overcrowding', 'aspire_buy_land', 'hh_tenure_type',
             'rent_negotiated', 'near_waste_dump', 'is_slum']
FIX_CHECK = ['dist_primary_school_m', 'access_mobility_rating',
             'access_smoke_alarms', 'access_kitchen_accommodations',
             'govt_provides_street_lighting', 'govt_functional_recreation',
             'n_dwelling_units_occupied', 'dwelling_tenure_type',
             'hh_head_sex', 'is_urban']

still_present = [c for c in BUG_CHECK if c in master_clean.columns]
missing_fixes = [c for c in FIX_CHECK if c not in master_clean.columns]

assert not still_present, f"Old wrong names still present: {still_present}"
assert not missing_fixes, f"Expected fixed columns missing: {missing_fixes}"
print("✓  Bug check passed — old wrong names gone, fixed columns present.")
print(f"✓  hh_head_sex coverage: {master_clean['hh_head_sex'].notna().mean()*100:.1f}%")


# Sentinel cleanup check (added alongside label-bug check above)
SENTINEL_MAX = {
    'dwelling_yr_built': 1900,   # must be >= this after cleanup
}
if 'dwelling_yr_last_renovated' in master_clean.columns:
    assert (master_clean['dwelling_yr_last_renovated'] != -1).all(), \
        "Sentinel -1 still present in dwelling_yr_last_renovated"
if 'security_charge_kes' in master_clean.columns:
    assert (master_clean['security_charge_kes'].dropna() >= 0).all(), \
        "Negative security_charge_kes still present"
if 'dwelling_yr_built' in master_clean.columns:
    _bad = master_clean['dwelling_yr_built'].dropna()
    assert ((_bad >= 1900) | _bad.isna()).all(), "Invalid dwelling_yr_built still present"
print("✓  Sentinel cleanup check passed.")


✓  Bug check passed — old wrong names gone, fixed columns present.
✓  hh_head_sex coverage: 99.7%
✓  Sentinel cleanup check passed.


In [24]:
# ── 3. Save (only runs if the pre-save verification above passed) ────────────
master_clean.to_parquet(PQ / 'master_frame.parquet')
print("\n✓  master_frame.parquet saved with clean column names.")



✓  master_frame.parquet saved with clean column names.


In [25]:
# ── Sanity check the patch before saving ──────────────────────────────────
check_cols = ['g01a','g01d','h01','h04','g03','g04']
for raw in check_cols:
    if raw in master.columns:
        print(f'{raw:8s} -> {RENAME_MAP[raw]:28s} range={sorted(master[raw].dropna().unique())[:6]}')

g01a     -> dist_primary_school_m        range=[np.float64(0.0), np.float64(0.1), np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0)]
g01d     -> dist_health_facility_m       range=[np.float64(0.0), np.float64(0.2), np.float64(0.3), np.float64(0.6), np.float64(1.0), np.float64(2.0)]
h01      -> access_mobility_rating       range=[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
h04      -> access_wide_doorways         range=[np.float64(1.0), np.float64(2.0), np.float64(3.0)]
g03      -> road_surface_type            range=[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
g04      -> road_has_street_lights       range=[np.float64(0.0), np.float64(1.0)]
